# Football Analysis Pipeline

---

Analyze broadcast football footage using computer vision and machine learning. This pipeline detects players, goalkeepers, referees, and the ball, tracks their movements, classifies teams by jersey color, and generates annotated video output.

![Football AI](https://media.roboflow.com/notebooks/examples/football-ai-diagram.png)

### What This Notebook Does

| Stage | Technology | Output |
|-------|------------|--------|
| **Detection** | YOLOv8 | Players, GK, referees, ball |
| **Tracking** | ByteTrack | Unique IDs across frames |
| **Team Classification** | SigLIP + KMeans | Team colors (red vs blue) |
| **Ball Tracking** | Slicer + Kalman | Smooth ball trajectory |
| **Output** | Supervision | Annotated video file |

## Before You Start

### Select GPU Runtime

**Note:** Processing is 10-50x faster with GPU. Navigate to `Runtime` > `Change runtime type` > `Hardware accelerator` > `GPU (T4)` and click `Save`.

In [ ]:
!nvidia-smi

## Install Dependencies

**Note:** This cell clones the repository, installs packages, and downloads pre-trained models. Takes 2-3 minutes on first run.

**Important:** After this cell completes, you may need to restart the runtime (`Runtime` > `Restart runtime`) if you see numpy errors.

In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/esharif20/Spatio-Temporal-GNN-Football-Analysis.git"
REPO_DIR = "/content/football_analysis"

os.chdir("/content")
if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

print("Cloning repository...")
!git clone --quiet {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)

# Fix numpy version FIRST (Colab has numpy 2.x which breaks supervision)
print("Fixing numpy version...")
!pip install -q "numpy<2"

print("Installing dependencies...")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
!bash colab_setup.sh 2>&1 | grep -E "(Downloading|Installing|Available|error|Error)" | head -20

print("\nSetup complete!")
print("\nIf you see numpy errors later, restart runtime: Runtime > Restart runtime")

### Verify Environment

**Note:** Check that GPU, models, and sample videos are available.

In [ ]:
import torch
from pathlib import Path

print("=" * 50)
print("ENVIRONMENT")
print("=" * 50)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\nModels:")
for name, path in [("Player", "src/models/player_detection.pt"),
                   ("Ball", "src/models/ball_detection.pt"),
                   ("Pitch", "src/models/pitch_detection.pt")]:
    status = "OK" if Path(path).exists() else "MISSING"
    print(f"  {name}: {status}")

print("\nSample Videos:")
videos = sorted(Path("src/input_videos").glob("*.mp4"))
for v in videos[:5]:
    size = v.stat().st_size / (1024*1024)
    print(f"  {v.stem} ({size:.1f} MB)")

---

## Pipeline Configuration

### Pipeline Modes

| Mode | What It Does | Speed |
|------|--------------|-------|
| `all` | Detection + Tracking + Teams + Ball | Slowest |
| `team` | Detection + Tracking + Team colors | Medium |
| `track` | Detection + Tracking (no team colors) | Medium |
| `players` | Detection only (bounding boxes) | Fast |
| `ball` | Ball detection and tracking only | Medium |
| `pitch` | Pitch keypoint detection | Fast |
| `radar` | 2D tactical top-down view | Slowest |

**Note:** Start with `all` mode for complete analysis, or `players` for a quick test.

In [ ]:
#@title Configuration { display-mode: "form" }

#@markdown ### Video Selection
CLIP = "0bfacc_0"  #@param {type:"string"}

#@markdown ### Pipeline Mode
MODE = "all"  #@param ["all", "team", "track", "players", "ball", "pitch", "radar"]

#@markdown ### Ball Tracking
BALL_CONF = 0.15  #@param {type:"slider", min:0.05, max:0.5, step:0.05}
FAST_BALL = False  #@param {type:"boolean"}

#@markdown ### Processing
FRESH = True  #@param {type:"boolean"}

print(f"Clip: {CLIP}")
print(f"Mode: {MODE}")
print(f"Ball conf: {BALL_CONF}")
print(f"Fast ball: {FAST_BALL}")
print(f"Fresh run: {FRESH}")

---

## Run Pipeline

**Note:** Processing time depends on video length and mode. Expect 2-5 minutes for a 30-second clip with `all` mode.

In [ ]:
import os
import torch
from pathlib import Path

# Validate video exists
clip_path = Path(f"src/input_videos/{CLIP}.mp4")
if not clip_path.exists():
    available = [p.stem for p in Path("src/input_videos").glob("*.mp4")]
    raise FileNotFoundError(f"Video '{CLIP}' not found.\nAvailable: {available}")

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build command
cmd = f"DEVICE={device} bash src/run.sh {MODE} {CLIP}"
if FRESH:
    cmd += " --fresh"
if FAST_BALL:
    cmd += " --fast-ball"
else:
    cmd += f" --ball-conf {BALL_CONF}"

print("=" * 60)
print(f"Video:   {CLIP}")
print(f"Mode:    {MODE}")
print(f"Device:  {device}")
print(f"Command: {cmd}")
print("=" * 60 + "\n")

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
!{cmd}

# Set output path
OUTPUT_VIDEO = f"src/output_videos/{CLIP}/{CLIP}_{MODE.upper()}.mp4"
INPUT_VIDEO = f"src/input_videos/{CLIP}.mp4"

if Path(OUTPUT_VIDEO).exists():
    size = Path(OUTPUT_VIDEO).stat().st_size / (1024*1024)
    print(f"\nOutput saved: {OUTPUT_VIDEO} ({size:.1f} MB)")
else:
    print(f"\nWarning: Output not found at {OUTPUT_VIDEO}")

---

## Visualize Results

### Preview Output Video

**Note:** Displays the processed video directly in the notebook using HTML5 video player.

In [ ]:
from IPython.display import HTML
from pathlib import Path
import base64

def show_video(path, width=800):
    """Display video in notebook using base64 encoding."""
    p = Path(path)
    if not p.exists():
        print(f"Video not found: {path}")
        return None
    
    size = p.stat().st_size / (1024*1024)
    print(f"Loading: {p.name} ({size:.1f} MB)")
    
    with open(p, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    
    return HTML(f'''<video width="{width}" controls>
        <source src="data:video/mp4;base64,{data}" type="video/mp4">
    </video>''')

show_video(OUTPUT_VIDEO)

### Sample Frames

**Note:** Displays individual frames from the output video for closer inspection.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

def get_frame(video_path, frame_idx=0):
    """Extract a single frame from video."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None

if Path(OUTPUT_VIDEO).exists():
    cap = cv2.VideoCapture(OUTPUT_VIDEO)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    # Get frames at 0%, 33%, 66% of video
    indices = [0, total//3, 2*total//3]
    frames = [get_frame(OUTPUT_VIDEO, i) for i in indices]
    frames = [f for f in frames if f is not None]
    
    if frames:
        fig, axes = plt.subplots(1, len(frames), figsize=(16, 5))
        for ax, frame, idx in zip(axes, frames, indices):
            ax.imshow(frame)
            ax.set_title(f"Frame {idx}")
            ax.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("Output video not found. Run the pipeline first.")

### Input vs Output Comparison

**Note:** Side-by-side comparison of original and processed frames.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

FRAME_IDX = 100

if Path(INPUT_VIDEO).exists() and Path(OUTPUT_VIDEO).exists():
    input_frame = get_frame(INPUT_VIDEO, FRAME_IDX)
    output_frame = get_frame(OUTPUT_VIDEO, FRAME_IDX)
    
    if input_frame is not None and output_frame is not None:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        ax1.imshow(input_frame)
        ax1.set_title("Original")
        ax1.axis("off")
        ax2.imshow(output_frame)
        ax2.set_title("Processed")
        ax2.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("Videos not found.")

---

## Detection Statistics

**Note:** Load cached detection data and display tracking statistics.

In [ ]:
import pickle
from pathlib import Path

def load_stats(clip_name):
    """Load and display tracking statistics from cache."""
    stub_dir = Path("src/stubs")
    
    print("=" * 50)
    print("DETECTION STATISTICS")
    print("=" * 50)
    
    # People tracks
    people_stub = stub_dir / f"{clip_name}_people_tracks.pkl"
    if people_stub.exists():
        with open(people_stub, "rb") as f:
            people = pickle.load(f)
        
        for category in ["players", "goalkeepers", "referees"]:
            if category in people:
                ids = set()
                detections = 0
                for frame_data in people[category]:
                    ids.update(frame_data.keys())
                    detections += len(frame_data)
                print(f"\n{category.capitalize()}:")
                print(f"  Unique IDs: {len(ids)}")
                print(f"  Total detections: {detections}")
    else:
        print("\nNo people tracks found.")
    
    # Ball tracks
    ball_stub = stub_dir / f"{clip_name}_ball_tracks.pkl"
    if ball_stub.exists():
        with open(ball_stub, "rb") as f:
            ball = pickle.load(f)
        if "ball" in ball:
            ball_data = ball["ball"]
            detected = sum(1 for f in ball_data if f)
            total = len(ball_data)
            pct = 100 * detected / total if total > 0 else 0
            print(f"\nBall:")
            print(f"  Detected: {detected}/{total} frames ({pct:.1f}%)")
    
    print("\n" + "=" * 50)

try:
    load_stats(CLIP)
except Exception as e:
    print(f"Could not load stats: {e}")

---

## Download Output

In [ ]:
from google.colab import files
from pathlib import Path

if Path(OUTPUT_VIDEO).exists():
    size = Path(OUTPUT_VIDEO).stat().st_size / (1024*1024)
    print(f"Downloading: {Path(OUTPUT_VIDEO).name} ({size:.1f} MB)")
    files.download(OUTPUT_VIDEO)
else:
    print("No output video to download.")

---

## Upload Your Own Video

**Note:** Upload a custom video file (MP4 recommended). After uploading, update `CLIP` in the Configuration section and re-run the pipeline.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

print("Select a video file:")
uploaded = files.upload()

for filename in uploaded.keys():
    dest = Path(f"src/input_videos/{filename}")
    shutil.move(filename, dest)
    size = dest.stat().st_size / (1024*1024)
    print(f"\nUploaded: {filename} ({size:.1f} MB)")
    print(f"\nSet CLIP = \"{dest.stem}\" in Configuration and re-run.")

---

## Reference

### CLI Options

| Option | Default | Description |
|--------|---------|-------------|
| `--fresh` | off | Ignore cache, reprocess everything |
| `--fast-ball` | off | Faster ball tracking (less accurate) |
| `--ball-conf` | 0.15 | Ball detection confidence (lower = more detections) |
| `--ball-kalman` | off | Enable Kalman smoothing for ball trajectory |
| `--no-ball-model` | off | Use multi-class model instead of dedicated ball model |

### Troubleshooting

| Problem | Solution |
|---------|----------|
| No GPU available | `Runtime` > `Change runtime type` > `GPU` |
| `numpy.strings` error | `Runtime` > `Restart runtime`, then re-run from Verify Environment |
| Video not found | Check CLIP name matches file in `src/input_videos/` |
| Models missing | Re-run the Install Dependencies cell |
| Ball not detected | Lower `BALL_CONF` to 0.10 |
| Out of memory | Use `--fast-ball` or shorter video clips |

### Links

- [GitHub Repository](https://github.com/esharif20/Spatio-Temporal-GNN-Football-Analysis)
- [YOLOv8 Documentation](https://docs.ultralytics.com/)
- [Supervision Library](https://supervision.roboflow.com/)